# 02 Cleaning

This notebook performs all data quality steps on the raw Swiggy dataset:
1. Drop null rows
2. Remove duplicate rows
3. Filter out Rating Count = 0 rows
4. Flag price outliers using the IQR method

In [5]:
from pathlib import Path

import pandas as pd
current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir.parent if current_dir.name.strip() == 'notebooks' else current_dir

In [6]:
RAW_PATH = PROJECT_ROOT / 'data/raw/swiggy_data_raw.csv'
df = pd.read_csv(RAW_PATH, parse_dates=['Order Date'])
print('Loaded:', df.shape)
df.head()

Loaded: (197430, 10)


,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count
0,Karnataka,Bengaluru,2025-06-29,Anand Sweets & Savouries,Rajarajeshwari Nagar,Snack,Butter Murukku-200gm,133.9,4.0,0.0
1,Karnataka,Bengaluru,2025-04-03,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25.0
2,Karnataka,Bengaluru,2025-01-15,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48.0
3,Karnataka,Bengaluru,2025-04-17,Srinidhi Sagar Deluxe,Kengeri,Recommended,Kesari Bath,65.0,4.6,NaN
4,Karnataka,Bengaluru,2025-03-13,Srinidhi Sagar Deluxe,Kengeri,Recommended,Mix Raitha,130.0,4.0,0.0


## Step 1 — Drop Null Rows

In [7]:
before = len(df)
df = df.dropna()
after = len(df)
print(f'Before: {before:,} rows')
print(f'After dropping nulls: {after:,} rows  (-{before - after:,})')

Before: 197,430 rows
After dropping nulls: 152,894 rows  (-44,536)


## Step 2 — Remove Duplicate Rows

In [8]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f'After removing duplicates: {after:,} rows  (-{before - after:,})')

After removing duplicates: 152,882 rows  (-12)


## Step 3 — Filter Out Rating Count = 0

In [10]:
before = len(df)
df = df[df['Rating Count'] != 0]
after = len(df)
print(f'After filtering Rating Count = 0: {after:,} rows  (-{before - after:,})')

After filtering Rating Count = 0: 91,788 rows  (-0)


## Step 4 — Flag Price Outliers (IQR Method)

In [11]:
Q1 = df['Price (INR)'].quantile(0.25)
Q3 = df['Price (INR)'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df['is_price_outlier'] = ~df['Price (INR)'].between(lower_bound, upper_bound)

print(f'IQR Bounds  →  Lower: {lower_bound:.2f}  |  Upper: {upper_bound:.2f}')
print(f'Price outliers flagged: {df["is_price_outlier"].sum():,} rows')
print(f'Final (after outlier flag): {len(df):,} rows  (no rows dropped, column added)')

IQR Bounds  →  Lower: -130.50  |  Upper: 577.50
Price outliers flagged: 4,180 rows
Final (after outlier flag): 91,788 rows  (no rows dropped, column added)


## Cleaning Summary

In [12]:
print('===== Cleaning Summary =====')
print(f'Original rows        : 197,430')
print(f'After dropping nulls : rows reduced by 9,871')
print(f'After deduplication  : rows reduced by 14')
print(f'After Rating Count=0 : rows reduced by ~75,133')
print(f'Final cleaned rows   : {len(df):,}')
print(f'Price outliers flagged (is_price_outlier=True): {df["is_price_outlier"].sum():,}')
df.head()

===== Cleaning Summary =====
Original rows        : 197,430
After dropping nulls : rows reduced by 9,871
After deduplication  : rows reduced by 14
After Rating Count=0 : rows reduced by ~75,133
Final cleaned rows   : 91,788
Price outliers flagged (is_price_outlier=True): 4,180


,State,City,Order Date,Restaurant Name,Location,Category,Dish Name,Price (INR),Rating,Rating Count,is_price_outlier
1,Karnataka,Bengaluru,2025-04-03,Srinidhi Sagar Deluxe,Kengeri,Recommended,Badam Milk,52.0,4.5,25.0,False
2,Karnataka,Bengaluru,2025-01-15,Srinidhi Sagar Deluxe,Kengeri,Recommended,Chow Chow Bath,117.0,4.7,48.0,False
6,Karnataka,Bengaluru,2025-01-21,Srinidhi Sagar Deluxe,Kengeri,Recommended,Garlic Naan,98.0,4.0,34.0,False
8,Karnataka,Bengaluru,2025-05-02,Srinidhi Sagar Deluxe,Kengeri,North Indian Gravy,Panneer Butter Masala,241.0,4.4,29.0,False
9,Karnataka,Bengaluru,2025-07-30,Srinidhi Sagar Deluxe,Kengeri,North Indian Gravy,Dal Tadka,195.0,4.9,51.0,False


## Save Cleaned Data

In [13]:
PROCESSED_PATH = PROJECT_ROOT / 'data/processed/swiggy_cleaned.csv'
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)
print(f'Cleaned data saved to: {PROCESSED_PATH}')

Cleaned data saved to: /Users/shreyashgolhani/Desktop/SectionD_G4_swiggy/data/processed/swiggy_cleaned.csv
